# Required Capstone Assignment 20.1: Initial Report and Exploratory Data Analysis (EDA)

# In this module, you will work on performing exploratory data analysis (EDA) to develop an initial report for your capstone project. You will use EDA to see what data can reveal beyond the formal modeling, hypothesis testing task, and data training to provide a better understanding of dataset variables and the relationships between them. You are encouraged to spend your time in this module cleaning your data and use feature engineering and EDA techniques to create visualizations to make sense of your findings. Additionally, you will also be required to use one of the ML algorithms you have learned so far in the program to develop a baseline model to use as a comparison in Module 24. You will have time in Module 24 to include additional models, clean the code, and make your work presentable for technical and non-technical audiences. For now, you will do the ‘heavy lifting’ of finding the answer to your research question.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set visualization style
sns.set(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('../data/online_retail_II.xlsx')

print(f"Initial Shape: {df.shape}")
df.head()

In [ ]:
## 1. Data Cleaning

# Drop duplicates
df = df.drop_duplicates()

# Drop rows with missing Customer ID (we cannot segment unknown customers)
df = df.dropna(subset=['Customer ID'])

# Handle Returns (Negative Quantity)
# We typically remove cancelled orders for segmentation or handle them separately
df = df[(df['Quantity'] > 0)]

# Create 'TotalPrice' Feature (Needed for Monetary value)
df['TotalPrice'] = df['Quantity'] * df['Price']

print(f"Shape after cleaning: {df.shape}")
print("Missing Values Check:")
print(df.isnull().sum())

In [ ]:
## 2. Feature Engineering: RFM Metrics

# Set snapshot date (1 day after the last invoice)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# Group by Customer ID to calculate Recency, Frequency, Monetary
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'Invoice': 'count',                                      # Frequency
    'TotalPrice': 'sum'                                      # Monetary
})

# Rename columns
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'Invoice': 'Frequency',
    'TotalPrice': 'Monetary'
}, inplace=True)

print(f"RFM Table Shape: {rfm.shape}")
rfm.head()

In [ ]:
## 3. Exploratory Data Analysis (EDA)

# Visualize Distributions
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(rfm['Recency'], kde=True)
plt.title('Recency Distribution')

plt.subplot(1, 3, 2)
# Frequency is usually heavily skewed, using log scale helps visualization
sns.histplot(rfm['Frequency'], kde=True, log_scale=True)
plt.title('Frequency Distribution (Log)')

plt.subplot(1, 3, 3)
sns.histplot(rfm['Monetary'], kde=True, log_scale=True)
plt.title('Monetary Distribution (Log)')

plt.tight_layout()
plt.show()

# Outlier Analysis using Boxplots
plt.figure(figsize=(10, 6))
sns.boxplot(data=rfm)
plt.title('Outlier Analysis (Raw Data)')
plt.yscale('log') # Log scale to handle massive monetary outliers
plt.show()

In [ ]:
## 4. Preprocessing for Modeling

# Unskew the data (Log Transformation)
# K-Means assumes normal distribution, but RFM is usually right-skewed.
rfm_log = np.log1p(rfm)

# Scale the data (StandardScaler)
# K-Means is distance-based, so variables must be on the same scale.
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled = pd.DataFrame(rfm_scaled, columns=rfm.columns, index=rfm.index)

print("Data Scaled and Ready.")
rfm_scaled.head()

In [ ]:
## 5. Baseline Model: K-Means Clustering

# Define Baseline: K=3 (Arbitrary start)
kmeans_baseline = KMeans(n_clusters=3, random_state=42)
rfm['Cluster_Baseline'] = kmeans_baseline.fit_predict(rfm_scaled)

# Evaluation Metric: Silhouette Score
# Rationale: Silhouette Score measures how similar an object is to its own cluster 
# compared to other clusters. Range is [-1, 1]. Higher is better.
score = silhouette_score(rfm_scaled, rfm['Cluster_Baseline'])

print(f"Baseline Silhouette Score (K=3): {score:.4f}")

# Visualize Baseline Clusters
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Recency', y='Monetary', hue='Cluster_Baseline', data=rfm, palette='viridis')
plt.title('Baseline Clusters: Recency vs Monetary')
plt.yscale('log')
plt.show()